In [1]:
import sys
sys.path.insert(0, '/home/scanbot/ua_tensors')

import numpy as np
import matplotlib.pyplot as plt
from embed import Embed
import requests

T = Embed(temp=0.0)

from attention import Attention
from algebra import Abs, Sum

In [2]:
url  = "https://raw.githubusercontent.com/mledoze/countries/master/countries.json"
data = requests.get(url).json()

# --- build entity lists ---
countries, regions, subregions, languages = [], [], [], []
triples = []

cca3_to_name = {e['cca3']: e.get('name', {}).get('common', '') for e in data}

for entry in data:
    country   = entry.get('name', {}).get('common', '')
    region    = entry.get('region', '')
    subregion = entry.get('subregion', '')
    langs     = list(entry.get('languages', {}).values())
    borders   = entry.get('borders', [])  # list of cca3 codes
    currencies = list(entry.get('currencies', {}).keys())
    timezones  = entry.get('timezones', [])

    if not (country and region):
        continue
    countries.append(country)
    regions.append(region)
    if subregion:
        subregions.append(subregion)
    languages.extend(langs)
    triples.append((country, 'region',    region))
    triples.append((country, 'subregion', subregion) if subregion else None)
    for lang in langs:
        triples.append((country, 'speaks', lang))
    for b in borders:
        triples.append((country, 'borders', cca3_to_name.get(b, '')))
    for c in currencies:
        triples.append((country, 'currency', c))
    for tz in timezones:
        triples.append((country, 'timezone', tz))

triples = [t for t in triples if t]

countries   = sorted(set(countries))
regions     = sorted(set(regions))
subregions  = sorted(set(subregions))
languages   = sorted(set(languages))
borders_list  = sorted(set(countries))   # country→country
currencies    = sorted(set(t[2] for t in triples if t[1] == 'currency'))
timezones     = sorted(set(t[2] for t in triples if t[1] == 'timezone'))

c_idx = {c: i for i, c in enumerate(countries)}
r_idx = {r: i for i, r in enumerate(regions)}
s_idx = {s: i for i, s in enumerate(subregions)}
l_idx = {l: i for i, l in enumerate(languages)}
cu_idx = {c: i for i, c in enumerate(currencies)}
tz_idx = {t: i for i, t in enumerate(timezones)}

NC, NR, NS, NL = len(countries), len(regions), len(subregions), len(languages)
print(f"Countries: {NC}, Regions: {NR}, Subregions: {NS}, Languages: {NL}")

# --- build relation matrices ---
R_loc    = np.zeros((NC, NR))   # country → region   (binary)
R_sub    = np.zeros((NC, NS))   # country → subregion
R_speaks = np.zeros((NC, NL))   # country → language
R_borders  = np.zeros((NC, NC))    # country × country
R_currency = np.zeros((NC, len(currencies)))
R_timezone = np.zeros((NC, len(timezones)))

for t in triples:
    country, rel, target = t
    if country not in c_idx:
        continue
    ci = c_idx[country]
    if rel == 'region'    and target in r_idx: R_loc[ci,    r_idx[target]] = 1.0
    if rel == 'subregion' and target in s_idx: R_sub[ci,    s_idx[target]] = 1.0
    if rel == 'speaks'    and target in l_idx: R_speaks[ci, l_idx[target]] = 1.0
    if rel == 'borders'  and target in c_idx:  R_borders[ci,  c_idx[target]]  = 1.0
    if rel == 'currency' and target in cu_idx: R_currency[ci, cu_idx[target]] = 1.0
    if rel == 'timezone' and target in tz_idx: R_timezone[ci, tz_idx[target]] = 1.0

print(f"R_loc    shape: {R_loc.shape}   density: {R_loc.mean():.3f}")
print(f"R_sub    shape: {R_sub.shape}  density: {R_sub.mean():.3f}")
print(f"R_speaks shape: {R_speaks.shape}  density: {R_speaks.mean():.3f}")

Countries: 250, Regions: 6, Subregions: 24, Languages: 155
R_loc    shape: (250, 6)   density: 0.167
R_sub    shape: (250, 24)  density: 0.041
R_speaks shape: (250, 155)  density: 0.011


In [3]:
# Updated Cell 4 logic to ensure 2D outputs
emb_loc,      EmbR_loc,      rep_loc      = T.ConceptEmbed(R_loc,      temp=0.1)
emb_sub,      EmbR_sub,      rep_sub      = T.ConceptEmbed(R_sub,      temp=0.1)
emb_speaks,   EmbR_speaks,   rep_speaks   = T.ConceptEmbed(R_speaks,   temp=0.1)
emb_borders,  EmbR_borders,  rep_borders  = T.ConceptEmbed(R_borders,  temp=0.1)
emb_currency, EmbR_currency, rep_currency = T.ConceptEmbed(R_currency, temp=0.1)
emb_timezone, EmbR_timezone, rep_timezone = T.ConceptEmbed(R_timezone, temp=0.1)

# FORCE RE-SHAPE: This ensures that even if a variable came back 1D, 
# it is forced back into a 2D column/matrix so the downstream cells don't crash.
emb_loc      = np.atleast_2d(emb_loc)
emb_sub      = np.atleast_2d(emb_sub)
emb_speaks   = np.atleast_2d(emb_speaks)
emb_borders  = np.atleast_2d(emb_borders)
emb_currency = np.atleast_2d(emb_currency)
emb_timezone = np.atleast_2d(emb_timezone)

# Now your original horizontal stack will also be safe
emb = np.hstack([emb_loc, emb_sub, emb_speaks, emb_borders, emb_currency, emb_timezone])

d0 = emb_loc.shape[1]
d1 = d0 + emb_sub.shape[1]
d2 = d1 + emb_speaks.shape[1]
d3 = d2 + emb_borders.shape[1]
d4 = d3 + emb_currency.shape[1]
d5 = d4 + emb_timezone.shape[1]

head_cols = [
    np.arange(0,  d0),
    np.arange(d0, d1),
    np.arange(d1, d2),
    np.arange(d2, d3),
    np.arange(d3, d4),
    np.arange(d4, d5),
]

concept_labels = (
    [('loc',      regions[c])    for c in rep_loc]      +
    [('sub',      subregions[c]) for c in rep_sub]      +
    [('speaks',   languages[c])  for c in rep_speaks]   +
    [('borders',  countries[c])  for c in rep_borders]  +
    [('currency', currencies[c]) for c in rep_currency] +
    [('timezone', timezones[c])  for c in rep_timezone]
)

/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.11/dist-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [4]:
print("embedded continents ", emb_loc.shape)
print("embedded subregions ", emb_sub.shape)
print("embedded languages ", emb_speaks.shape)

embedded continents  (250, 6)
embedded subregions  (250, 24)
embedded languages  (250, 150)


In [5]:
mem_loc      = Attention(emb_loc,      temp=0.9)
mem_sub      = Attention(emb_sub,      temp=0.9)
mem_speaks   = Attention(emb_speaks,   temp=0.9)

In [6]:
def decode(mha, labels):
    extent = sorted([(countries[i], mha.fp.state[i]) for i in np.where(mha.fp.state > mha.eps)[0]], key=lambda x: -x[1])
    print("Extent:", [(c, f"{s:.3f}") for c, s in extent])
    for name, (intent, _) in mha.intents.items():
        label_arr, rep_cols = labels[name]
        concepts = sorted([(label_arr[rep_cols[i]], intent[i]) for i in range(len(rep_cols)) if intent[i] > mha.eps], key=lambda x: -x[1])
        print(f"Intent [{name}]:", [(c, f"{s:.3f}") for c, s in concepts])

In [7]:
def decode(mha, labels):
    extent = sorted([(countries[i], mha.fp.state[i]) for i in np.where(mha.fp.state > mha.eps)[0]], key=lambda x: -x[1])
    print("Extent:", [(c, f"{s:.3f}") for c, s in extent])
    for name, (intent, _) in mha.intents.items():
        label_arr, rep_cols = labels[name]
        concepts = sorted([(label_arr[rep_cols[i]], intent[i]) for i in range(len(rep_cols)) if intent[i] > mha.eps], key=lambda x: -x[1])
        print(f"Intent [{name}]:", [(c, f"{s:.3f}") for c, s in concepts])

In [ ]:
from functools import reduce
from attention import MultiHeadAttention

mha = MultiHeadAttention(
    heads = [mem_loc, mem_sub, mem_speaks],
    names = ['loc', 'sub', 'speaks'],
)


In [10]:
from explorer import CategoryExplorer

explorer = CategoryExplorer(mha, eps=1e-3)
emb_cat, EmbR, rep_cat = explorer.explore_lattice(len(countries))

print("Shape:", emb_cat.shape)


initial xploration complete
Shape: (250, 77)


In [26]:
col_to_key = {col: key for col, key in enumerate(explorer.categories.keys())}

In [29]:
print("emb_cat shape:", emb_cat.shape)
print("emb_speaks.T shape:", emb_speaks.T.shape)
print("result shape:", result.shape)
print("sample witnesses:", list(P._witnesses.items())[:3])

emb_cat shape: (250, 77)
emb_speaks.T shape: (150, 250)
result shape: (250, 250)
sample witnesses: [((np.int64(0), np.int64(151)), {0: np.float64(0.39999999990757684)}), ((np.int64(0), np.int64(204)), {0: np.float64(0.39999999990757684)}), ((np.int64(1), np.int64(1)), {1: np.float64(0.6666666666666701)})]


In [74]:
from provenance import Provenance

def decode_country(country_name, emb_cat, 
                   emb_loc, emb_sub, emb_speaks,
                   rep_loc, rep_sub, rep_speaks,
                   regions, subregions, languages, c_idx, temp=0.0):
    
    P = Provenance()
    P.tracking = True
    P._clear_witnesses()
    
    result_loc    = P.Join(emb_cat.T, emb_loc,    temp=temp)
    result_sub    = P.Join(emb_cat.T, emb_sub,    temp=temp)
    result_speaks = P.Join(emb_cat.T, emb_speaks, temp=temp)
    
    idx = c_idx[country_name]
    cat_row = emb_cat[idx]
    active_cats = np.where(cat_row > 0)[0]
    
    print(f"\n=== {country_name} ===")
    for k in active_cats:
        print(f"\nCategory {k} (membership: {cat_row[k]:.3f})")
        
        row = result_loc[k]
        active = np.where((row > 1e-6) & (row < 1e8))[0]
        if len(active):
            print(f"  Continent:  {[(regions[rep_loc[l]],    round(row[l], 3)) for l in active]}")
        
        row = result_sub[k]
        active = np.where((row > 1e-6) & (row < 1e8))[0]
        if len(active):
            print(f"  Subregion:  {[(subregions[rep_sub[l]], round(row[l], 3)) for l in active]}")
        
        row = result_speaks[k]
        active = np.where((row > 1e-6) & (row < 1e8))[0]
        if len(active):
            print(f"  Languages:  {[(languages[rep_speaks[l]], round(row[l], 3)) for l in active]}")

In [78]:
from provenance import Provenance

def decode_all_categories(emb_cat, emb_loc, emb_sub, emb_speaks,
                           rep_loc, rep_sub, rep_speaks,
                           regions, subregions, languages, temp=0.0):
    P = Provenance()
    P.tracking = True
    P._clear_witnesses()
    
    result_loc    = P.Join(emb_cat.T, emb_loc,    temp=temp)
    result_sub    = P.Join(emb_cat.T, emb_sub,    temp=temp)
    result_speaks = P.Join(emb_cat.T, emb_speaks, temp=temp)
    
    for k in range(emb_cat.shape[1]):
        print(f"\n=== Category {k} ===")
        
        row = result_loc[k]
        active = np.where((row > 1e-6) & (row < 1e8))[0]
        if len(active):
            print(f"  Continent:  {[(regions[rep_loc[l]], round(row[l], 3)) for l in active]}")
        
        row = result_sub[k]
        active = np.where((row > 1e-6) & (row < 1e8))[0]
        if len(active):
            print(f"  Subregion:  {[(subregions[rep_sub[l]], round(row[l], 3)) for l in active]}")
        
        row = result_speaks[k]
        active = np.where((row > 1e-6) & (row < 1e8))[0]
        if len(active):
            print(f"  Languages:  {[(languages[rep_speaks[l]], round(row[l], 3)) for l in active]}")

In [79]:
decode_all_categories(emb_cat, emb_loc, emb_sub, emb_speaks,
                           rep_loc, rep_sub, rep_speaks,
                           regions, subregions, languages, temp=0.0)


=== Category 0 ===
  Continent:  [('Asia', np.float64(0.4))]
  Subregion:  [('Southern Asia', np.float64(0.4))]
  Languages:  [('Dari', np.float64(0.4)), ('Pashto', np.float64(0.4)), ('Turkmen', np.float64(0.4))]

=== Category 1 ===
  Continent:  [('Europe', np.float64(0.667))]
  Subregion:  [('Southeast Europe', np.float64(0.667))]
  Languages:  [('Albanian', np.float64(0.667)), ('Bosnian', np.float64(0.333)), ('Croatian', np.float64(0.333)), ('Serbian', np.float64(0.667))]

=== Category 2 ===
  Continent:  [('Africa', np.float64(0.658))]
  Subregion:  [('Northern Africa', np.float64(0.658))]
  Languages:  [('Arabic', np.float64(0.658)), ('Berber', np.float64(0.658)), ('English', np.float64(0.658)), ('Hassaniya', np.float64(0.026)), ('Spanish', np.float64(0.026))]

=== Category 3 ===
  Continent:  [('Oceania', np.float64(0.489))]
  Subregion:  [('Polynesia', np.float64(0.489))]
  Languages:  [('Cook Islands Māori', np.float64(0.489)), ('English', np.float64(0.489)), ('French', np.flo

In [80]:
unique_rows, inverse = np.unique(emb_cat, axis=0, return_inverse=True)
print(unique_rows.shape)  # should be (101, 77)

(101, 77)


In [89]:
def decode_meta_categories(emb_cat, result_loc, result_sub, result_speaks,
                            rep_loc, rep_sub, rep_speaks,
                            regions, subregions, languages):
    
    unique_rows, inverse = np.unique(emb_cat, axis=0, return_inverse=True)
    print(f"{len(unique_rows)} meta-categories found")
    
    for m, row in enumerate(unique_rows):
        active_cats = np.where(row > 1e-6)[0]
        if len(active_cats) == 0:
            continue
        print(f"\n=== Meta-Category {m} ===")
        for k in active_cats:
            print(f"  Base category {k} (membership: {row[k]:.3f})")

In [90]:
decode_meta_categories(emb_cat, emb_loc, emb_sub, emb_speaks,
                            rep_loc, rep_sub, rep_speaks,
                            regions, subregions, languages)

101 meta-categories found

=== Meta-Category 1 ===
  Base category 76 (membership: 0.994)

=== Meta-Category 2 ===
  Base category 75 (membership: 0.994)

=== Meta-Category 3 ===
  Base category 74 (membership: 0.667)

=== Meta-Category 4 ===
  Base category 73 (membership: 0.667)

=== Meta-Category 5 ===
  Base category 72 (membership: 0.994)

=== Meta-Category 6 ===
  Base category 71 (membership: 0.997)

=== Meta-Category 7 ===
  Base category 69 (membership: 0.994)

=== Meta-Category 8 ===
  Base category 68 (membership: 0.994)

=== Meta-Category 9 ===
  Base category 67 (membership: 0.667)

=== Meta-Category 10 ===
  Base category 66 (membership: 0.994)

=== Meta-Category 11 ===
  Base category 65 (membership: 0.994)

=== Meta-Category 12 ===
  Base category 64 (membership: 0.997)

=== Meta-Category 13 ===
  Base category 63 (membership: 0.994)

=== Meta-Category 14 ===
  Base category 62 (membership: 0.994)

=== Meta-Category 15 ===
  Base category 61 (membership: 0.994)

=== Met

In [91]:
def isomorphic_meta_categories(emb_cat):
    
    unique_rows, inverse = np.unique(emb_cat, axis=0, return_inverse=True)
    
    # for each unique row, get the set of active (col, value) pairs
    # two meta-categories are isomorphic if their value multisets are identical
    # regardless of which columns they occupy
    
    from collections import defaultdict
    
    value_signature = {}
    groups = defaultdict(list)
    
    for m, row in enumerate(unique_rows):
        active = np.where(row > 0)[0]
        if len(active) == 0:
            continue
        # signature is the sorted tuple of values only — not column indices
        sig = tuple(sorted(row[active].tolist()))
        groups[sig].append((m, active, row[active]))
    
    print("Isomorphic meta-category groups:\n")
    for sig, members in groups.items():
        if len(members) < 2:
            continue
        print(f"Signature: {sig}")
        for m, cols, vals in members:
            print(f"  Meta-category {m}: columns {cols.tolist()} at values {vals.tolist()}")
        print()

In [92]:
isomorphic_meta_categories(emb_cat)

Isomorphic meta-category groups:

Signature: (0.9944863482894747,)
  Meta-category 1: columns [76] at values [0.9944863482894747]
  Meta-category 2: columns [75] at values [0.9944863482894747]
  Meta-category 5: columns [72] at values [0.9944863482894747]
  Meta-category 7: columns [69] at values [0.9944863482894747]
  Meta-category 8: columns [68] at values [0.9944863482894747]
  Meta-category 10: columns [66] at values [0.9944863482894747]
  Meta-category 11: columns [65] at values [0.9944863482894747]
  Meta-category 13: columns [63] at values [0.9944863482894747]
  Meta-category 14: columns [62] at values [0.9944863482894747]
  Meta-category 15: columns [61] at values [0.9944863482894747]
  Meta-category 16: columns [60] at values [0.9944863482894747]
  Meta-category 17: columns [59] at values [0.9944863482894747]
  Meta-category 18: columns [58] at values [0.9944863482894747]
  Meta-category 19: columns [57] at values [0.9944863482894747]
  Meta-category 20: columns [56] at values

In [ ]:
import asyncio
import json

ANTHROPIC_API_KEY = "**************"


async def label_category(session, k, profile_loc, profile_sub, profile_speaks,
                          result_loc, result_sub, result_speaks,
                          rep_loc, rep_sub, rep_speaks,
                          regions, subregions, languages):
    
    row_loc    = result_loc[k]
    row_sub    = result_sub[k]
    row_speaks = result_speaks[k]
    
    def active(row, label_fn):
        idx = np.where((row > 1e-6) & (row < 1e8))[0]
        return [(label_fn(l), round(row[l], 4)) for l in idx]
    
    continents = active(row_loc,    lambda l: regions[rep_loc[l]])
    subregions_ = active(row_sub,   lambda l: subregions[rep_sub[l]])
    langs      = active(row_speaks, lambda l: languages[rep_speaks[l]])
    
    if not continents and not langs:
        return k, None
    
    prompt = f"""You are labeling a discovered geographic-linguistic category.

Continent: {continents}
Subregion: {subregions_}
Languages (name, strength): {sorted(langs, key=lambda x: -x[1])[:8]}

Give a short label (3-6 words) that best describes this category.
Respond with JSON only: {{"label": "..."}}"""

    response = await session.post(
        "https://api.anthropic.com/v1/messages",
        json={
            
            "model": "claude-sonnet-4-20250514",
            "max_tokens": 100,
            "messages": [{"role": "user", "content": prompt}]
        }
    )
    data = await response.json()
    text = data["content"][0]["text"].strip()
    result = json.loads(text)
    return k, result["label"]


async def label_all_categories(emb_cat, emb_loc, emb_sub, emb_speaks,
                                rep_loc, rep_sub, rep_speaks,
                                regions, subregions, languages, temp=0.0):
    import aiohttp
    
    P = Provenance()
    P.tracking = False
    result_loc    = P.Join(emb_cat.T, emb_loc,    temp=temp)
    result_sub    = P.Join(emb_cat.T, emb_sub,    temp=temp)
    result_speaks = P.Join(emb_cat.T, emb_speaks, temp=temp)
    
    async with aiohttp.ClientSession(headers={
        "x-api-key": ANTHROPIC_API_KEY,
        "anthropic-version": "2023-06-01",
        "content-type": "application/json"
    }) as session:
        tasks = [
            label_category(session, k, result_loc[k], result_sub[k], result_speaks[k],
                        result_loc, result_sub, result_speaks,
                        rep_loc, rep_sub, rep_speaks,
                        regions, subregions, languages)
            for k in range(emb_cat.shape[1])
        ]
        results = await asyncio.gather(*tasks)
    
    labels = {k: label for k, label in results if label is not None}
    return labels

# run
category_labels = await label_all_categories(
    emb_cat, emb_loc, emb_sub, emb_speaks,
    rep_loc, rep_sub, rep_speaks,
    regions, subregions, languages
)

for k, label in sorted(category_labels.items()):
    print(f"Category {k}: {label}")

Category 0: Afghan Language Group
Category 1: Western Balkans Language Region
Category 2: Northern African Arabic-Speaking Region
Category 3: Polynesian Languages and English
Category 4: Catalan-speaking Southern Europe
Category 5: Middle African Romance Languages
Category 6: Caribbean Multilingual Islands
Category 8: South American Indigenous Language Region
Category 9: Armenian-speaking Western Asia
Category 10: New Zealand Language Group
Category 11: Central European Austro-Bavarian Region
Category 12: Azerbaijani-Russian Western Asia Region
Category 13: Western Asian Multilingual Region
Category 14: Bengali-speaking Southern Asia
Category 15: Eastern European Slavic Region
Category 16: Western European Multilingual Region
Category 17: Belize Multilingual Region
Category 18: Francophone West Africa
Category 19: English-French North America
Category 20: Bhutan Dzongkha Speaking Region
Category 21: Southeast European Balkan Languages
Category 22: Namibian Multilingual Language Group
C

In [97]:
async def label_meta_category(session, m, row, category_labels):
    active_cats = np.where(row > 1e-6)[0]
    if len(active_cats) == 0:
        return m, "Uncategorized"
    
    components = [(category_labels.get(k, f"Category {k}"), round(row[k], 4)) 
                  for k in active_cats]
    components = sorted(components, key=lambda x: -x[1])
    
    prompt = f"""You are labeling a higher-order geographic-linguistic meta-category.
It is defined by graded membership in these base categories:
{components}

Give a short label (3-6 words) that best describes this combination.
Respond with JSON only: {{"label": "..."}}"""

    response = await session.post(
        "https://api.anthropic.com/v1/messages",
        json={
            "model": "claude-sonnet-4-20250514",
            "max_tokens": 100,
            "messages": [{"role": "user", "content": prompt}]
        }
    )
    data = await response.json()
    text = data["content"][0]["text"].strip()
    result = json.loads(text)
    return m, result["label"]


async def label_all_meta_categories(emb_cat, category_labels):
    import aiohttp
    
    unique_rows, inverse = np.unique(emb_cat, axis=0, return_inverse=True)
    
    async with aiohttp.ClientSession(headers={
        "x-api-key": ANTHROPIC_API_KEY,
        "anthropic-version": "2023-06-01",
        "content-type": "application/json"
    }) as session:
        tasks = [
            label_meta_category(session, m, row, category_labels)
            for m, row in enumerate(unique_rows)
        ]
        results = await asyncio.gather(*tasks)
    
    return {m: label for m, label in results if label is not None}

meta_category_labels = await label_all_meta_categories(emb_cat, category_labels)

for m, label in sorted(meta_category_labels.items()):
    print(f"Meta-category {m}: {label}")

Meta-category 0: Uncategorized
Meta-category 1: Vietnamese-speaking Southeast Asia
Meta-category 2: Ukrainian-speaking Eastern Europe
Meta-category 3: Turkish-speaking Western Asia
Meta-category 4: Lusophone Southeast Asian Region
Meta-category 5: Thai-speaking Southeast Asia
Meta-category 6: Norwegian Speaking Northern Europe
Meta-category 7: Central European Slovene Region
Meta-category 8: Romanian Southeast European Region
Meta-category 9: Portuguese-influenced Southern European areas
Meta-category 10: Polish Central European Region
Meta-category 11: Macedonian Southeast European Region
Meta-category 12: Korean-speaking Eastern Asia
Meta-category 13: Nepali-speaking Southern Asia
Meta-category 14: Burmese-speaking Southeast Asia
Meta-category 15: Montenegrin Southeast European Region
Meta-category 16: Eastern Asian Mongolian Region
Meta-category 17: Eastern European Moldavian Region
Meta-category 18: Maldivian Southern Asia
Meta-category 19: Lithuanian Northern Europe
Meta-category 

In [98]:
def isomorphic_meta_categories_labeled(emb_cat, meta_category_labels):
    
    unique_rows, inverse = np.unique(emb_cat, axis=0, return_inverse=True)
    
    from collections import defaultdict
    groups = defaultdict(list)
    
    for m, row in enumerate(unique_rows):
        active = np.where(row > 0)[0]
        sig = tuple(sorted(row[active].tolist()))
        groups[sig].append((m, active, row[active]))
    
    print("Isomorphic meta-category groups:\n")
    for sig, members in groups.items():
        if len(members) < 2:
            continue
        print(f"Signature: {sig}")
        for m, cols, vals in members:
            label = meta_category_labels.get(m, f"Meta-category {m}")
            print(f"  [{m}] {label}: columns {cols.tolist()} at values {vals.tolist()}")
        print()

isomorphic_meta_categories_labeled(emb_cat, meta_category_labels)

Isomorphic meta-category groups:

Signature: (0.9944863482894747,)
  [1] Vietnamese-speaking Southeast Asia: columns [76] at values [0.9944863482894747]
  [2] Ukrainian-speaking Eastern Europe: columns [75] at values [0.9944863482894747]
  [5] Thai-speaking Southeast Asia: columns [72] at values [0.9944863482894747]
  [7] Central European Slovene Region: columns [69] at values [0.9944863482894747]
  [8] Romanian Southeast European Region: columns [68] at values [0.9944863482894747]
  [10] Polish Central European Region: columns [66] at values [0.9944863482894747]
  [11] Macedonian Southeast European Region: columns [65] at values [0.9944863482894747]
  [13] Nepali-speaking Southern Asia: columns [63] at values [0.9944863482894747]
  [14] Burmese-speaking Southeast Asia: columns [62] at values [0.9944863482894747]
  [15] Montenegrin Southeast European Region: columns [61] at values [0.9944863482894747]
  [16] Eastern Asian Mongolian Region: columns [60] at values [0.9944863482894747]
  

In [104]:
async def label_isomorphic_groups(emb_cat, category_labels, meta_category_labels):
    import aiohttp
    from collections import defaultdict
    
    unique_rows, inverse = np.unique(emb_cat, axis=0, return_inverse=True)
    groups = defaultdict(list)
    
    for m, row in enumerate(unique_rows):
        active = np.where(row > 0)[0]
        sig = tuple(sorted(row[active].tolist()))
        groups[sig].append((m, active, row[active]))
    
    iso_groups = {sig: members for sig, members in groups.items() if len(members) >= 2}
    
    async def label_group(sig, members):
        member_descriptions = [
            meta_category_labels.get(m, f"Meta-category {m}")
            for m, cols, vals in members
        ]
        prompt = f"""These geographic-linguistic meta-categories are structurally isomorphic —
they share identical membership strengths {sig} but over different regions.

Members:
{chr(10).join(f"- {label}" for label in member_descriptions)}

What do all these have in common structurally? Give a single short label (3-6 words) 
describing the shared abstract pattern across all members.
Respond with JSON only, no markdown: {{"label": "..."}}"""

        response = await session.post(
            "https://api.anthropic.com/v1/messages",
            json={
                "model": "claude-sonnet-4-20250514",
                "max_tokens": 200,
                "messages": [{"role": "user", "content": prompt}]
            }
        )
        data = await response.json()
        text = data["content"][0]["text"].strip()
        text = text.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        try:
            result = json.loads(text)
            return sig, result["label"]
        except json.JSONDecodeError:
            print(f"Failed to parse: {repr(text[:200])}")
            return sig, None

    async with aiohttp.ClientSession(headers={
        "x-api-key": ANTHROPIC_API_KEY,
        "anthropic-version": "2023-06-01",
        "content-type": "application/json"
    }) as session:
        results = await asyncio.gather(*[
            label_group(sig, members) for sig, members in iso_groups.items()
        ])
    
    group_labels = {sig: label for sig, label in results if label is not None}
    
    print("Isomorphic meta-category groups:\n")
    for sig, members in iso_groups.items():
        print(f"Group: {group_labels.get(sig, 'Unlabeled')}")
        print(f"Signature: {sig}")
        for m, cols, vals in members:
            print(f"  [{m}] {meta_category_labels.get(m, f'Meta-category {m}')}")
        print()
    
    return group_labels

iso_labels = await label_isomorphic_groups(emb_cat, category_labels, meta_category_labels)

Isomorphic meta-category groups:

Group: Primary National Language Regions
Signature: (0.9944863482894747,)
  [1] Vietnamese-speaking Southeast Asia
  [2] Ukrainian-speaking Eastern Europe
  [5] Thai-speaking Southeast Asia
  [7] Central European Slovene Region
  [8] Romanian Southeast European Region
  [10] Polish Central European Region
  [11] Macedonian Southeast European Region
  [13] Nepali-speaking Southern Asia
  [14] Burmese-speaking Southeast Asia
  [15] Montenegrin Southeast European Region
  [16] Eastern Asian Mongolian Region
  [17] Eastern European Moldavian Region
  [18] Maldivian Southern Asia
  [19] Lithuanian Northern Europe
  [20] Latvian Northern European Region
  [21] Lao-speaking Southeast Asia
  [23] Japanese-speaking Eastern Asia
  [25] Persian-speaking Southern Asia
  [26] Indonesian-speaking Southeast Asia
  [29] Nordic Icelandic Region
  [30] Hungarian Central European Region
  [33] Arctic Greenlandic North America
  [36] Georgian-speaking Western Asia
  [41] 